# HF365 (HLS Spectral Indices) — Dataset Facts

Structural facts only: shape, missingness, timing, and value ranges for
the HF365 ground-validation dataset (EVI2, NDVI, NIRv across ~42 plots).
No correlation or modeling here -- see `02-correlation-analysis` for that.


In [ ]:
import os
import numpy as np
import pandas as pd

# vvv EDIT THIS to match your folder vvv
HF365_DIR = "."
# ^^^ EDIT THIS to match your folder ^^^

HF365_FILES = {
    "EVI2": "hf365-01-hls-evi2.csv",
    "NDVI": "hf365-02-hls-ndvi.csv",
    "NIRv": "hf365-03-hls-nirv.csv",
}


## 1. Load

In [ ]:
hf365_wide = {}
for name, fname in HF365_FILES.items():
    df = pd.read_csv(os.path.join(HF365_DIR, fname))
    df.columns = [c.strip() for c in df.columns]  # HF365's raw headers have stray spaces (e.g. "D1 ")
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    hf365_wide[name] = df
    print(f"{name}: {df.shape[0]} dates x {df.shape[1]} plots")


## 2. Shape and plot list

In [ ]:
for name, df in hf365_wide.items():
    print(f"{name}: {df.shape}")
print()
print(f"Plots (from NDVI): {list(hf365_wide['NDVI'].columns)}")
print(f"Plot count consistent across all 3 files: "
      f"{len(set(tuple(df.columns) for df in hf365_wide.values())) == 1}")


## 3. Date range

In [ ]:
for name, df in hf365_wide.items():
    print(f"{name}: {df.index.min().date()} to {df.index.max().date()}  "
          f"({df.index.nunique()} distinct dates)")

union_dates = sorted(set().union(*[set(df.index) for df in hf365_wide.values()]))
print(f"\nUnion of dates across all 3 files: {len(union_dates)}")
print(f"Union range: {union_dates[0].date()} to {union_dates[-1].date()}")


## 4. Missing values

Two levels: whole rows the file simply doesn't have (sparse by
construction -- HF365 only logs a row when at least one plot has data),
and individual blank cells within existing rows (per-plot cloud/snow
masking).

In [ ]:
span_days = (union_dates[-1] - union_dates[0]).days + 1
print(f"Calendar days spanned: {span_days}")
print(f"Distinct rows present: {len(union_dates)} ({100*len(union_dates)/span_days:.1f}% of calendar days)")
print("(Confirms rows only exist for actual overpass dates, not a padded daily calendar.)")


In [ ]:
for name, df in hf365_wide.items():
    total_cells = df.size
    na_cells = df.isna().sum().sum()
    all_na_rows = df.isna().all(axis=1).sum()
    print(f"{name}: {na_cells}/{total_cells} cells blank ({100*na_cells/total_cells:.1f}%), "
          f"rows with ALL plots blank: {all_na_rows}")


## 5. Timing between measurements

In [ ]:
for name, df in hf365_wide.items():
    diffs = pd.Series(df.index).diff().dt.days.dropna()
    print(f"{name}: median gap {diffs.median():.0f}d, mean {diffs.mean():.1f}d, "
          f"max {diffs.max():.0f}d, gaps>16d: {(diffs>16).sum()}")


## 6. Observations per calendar month

Whether coverage is seasonally even, or thinner in some months (e.g.
persistent winter cloud cover).

In [ ]:
for name, df in hf365_wide.items():
    n_valid_per_row = df.notna().sum(axis=1)
    by_month = n_valid_per_row.groupby(df.index.month).mean()
    print(f"{name} -- mean valid-plot-count by month:")
    print(by_month.round(1).to_string())
    print()


## 7. Per-plot completeness

In [ ]:
for name, df in hf365_wide.items():
    completeness = df.notna().mean(axis=0).sort_values()
    print(f"{name} -- least complete plots:")
    print((completeness.head(5) * 100).round(1).astype(str) + "%")
    print(f"{name} -- most complete plots:")
    print((completeness.tail(5) * 100).round(1).astype(str) + "%")
    print()


## 8. Cross-index consistency

Do the same dates/plots show up as valid across all three indices, or
does one index have systematically more gaps than the others?

In [ ]:
common_dates = set(hf365_wide["EVI2"].index) & set(hf365_wide["NDVI"].index) & set(hf365_wide["NIRv"].index)
print(f"Dates common to all 3 indices: {len(common_dates)}")
for name, df in hf365_wide.items():
    only_here = set(df.index) - common_dates
    print(f"Dates only in {name}: {len(only_here)}")


## 9. Value ranges (valid values only)

Sanity check against physically plausible bounds -- catches leftover
fill-value artifacts.

In [ ]:
expected_ranges = {"EVI2": (-1, 1), "NDVI": (-1, 1), "NIRv": (-1, 1)}
for name, df in hf365_wide.items():
    vals = df.values.flatten()
    vals = vals[~np.isnan(vals)]
    lo, hi = expected_ranges[name]
    n_out = ((vals < lo) | (vals > hi)).sum()
    print(f"{name}: min={vals.min():.3f}, max={vals.max():.3f}, "
          f"mean={vals.mean():.3f} -- {n_out} values outside plausible [{lo},{hi}] range")


## 10. Summary

In [ ]:
print("=== HF365 FACT SHEET ===")
for name, df in hf365_wide.items():
    diffs = pd.Series(df.index).diff().dt.days.dropna()
    na_pct = 100 * df.isna().sum().sum() / df.size
    print(f"{name}: {df.shape[0]} dates x {df.shape[1]} plots, "
          f"{df.index.min().date()} to {df.index.max().date()}, "
          f"{na_pct:.1f}% blank, median gap {diffs.median():.0f}d")
print(f"\nUnion dates across all 3 indices: {len(union_dates)} "
      f"({100*len(union_dates)/span_days:.1f}% of calendar days in range)")
